# Agentic RAG

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [26]:
from langchain_openai import ChatOpenAI
model_name = 'google/gemma-3-12b'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

# Agentic RAG Tool

In [22]:
from langchain_chroma import Chroma  # 向量資料庫 Chroma，用來儲存與檢索文件向量
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  # 文字嵌入模型
from langchain_core.messages import HumanMessage, SystemMessage  # LangChain 訊息格式，用於 LLM 對話
from langchain_core.tools import tool

# 載入文字嵌入模型
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-zh-v1.5")

# 載入向量資料庫
vector_store = Chroma(
    collection_name="clementshop_qa",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)


from pydantic import BaseModel, Field

class GetQAInput(BaseModel):
    question: str = Field(description="查詢的問題文字")
    k: int = Field(default=5, description="要回傳的文件數量")

@tool(args_schema=GetQAInput)
def get_qa(question, k=5):
    """根據使用者提出的問題，從向量資料庫中檢索出最相關的 K 筆問答文件。

    Args:
        question (str): 查詢的問題文字。
        k (int): 要回傳的文件數量，預設為 5。

    Returns:
        str: 檢索到的相似文件列表。
    """
    docs = vector_store.similarity_search(question, k=k)
    return str(docs)

get_qa.invoke({"question": "我要退貨", "k": 5})


"[Document(id='ac7482f8-e7f2-4ee3-a5ed-b27b8e37ebda', metadata={'source': 'qa_data\\\\QA_退貨.txt'}, page_content='Q10: 哪些商品不接受退貨？\\nA10: 客製化、易腐壞或已拆封之商品恕不接受退貨。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。'), Document(id='818269b7-4a2b-4690-9d09-13aed2a83bd9', metadata={'source': 'qa_data\\\\QA_退貨.txt'}, page_content='Q4: 退貨商品需要保持什麼狀態？\\nA4: 商品須保持全新未使用、包裝完整並附原始發票。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。\\n\\nQ5: 退貨申請審核多久？\\nA5: 客服將於收到申請後 3 個工作天內完成審核。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。'), Document(id='f207e8f7-9007-49ec-b327-28d98f340900', metadata={'source': 'qa_data\\\\QA_退貨.txt'}, page_content='Q3: 退貨需要支付運費嗎？\\nA3: 若為商品瑕疵或出貨錯誤，退貨運費由 ClementShop 負擔。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件

# 對話機器人

In [23]:
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage, AIMessageChunk
from langchain_core.output_parsers import StrOutputParser

class stream_chat_bot:
    def __init__(self, llm, tools):
        # 初始化對話機器人，傳入 LLM 與可用工具列表
        self.tools = tools
        # 將 LLM 綁定（bind）工具，使其具備自動呼叫工具的能力
        self.llm_with_tools = llm.bind_tools(tools)

        # 系統提示詞（System Prompt），用來設定 LLM 的角色與行為
        system_prompt = '''
你是一位專業的電商客服代表。
請根據規範，準確且禮貌地回答使用者的問題。  
請注意以下原則：
1. 回覆時不得提及或暗示任何內部文件、規章或系統資訊的存在。  
2. 僅能回應與公司平台、產品、訂單、客服流程等相關的問題。  
3. 若使用者的提問與平台或服務內容無關，請婉轉地表示無法協助，並引導回到與平台相關的主題。
4. 如果使用者的問題需要外部資料，請直接使用可用的工具完成查詢，不需向使用者確認。
5. 如果對話紀錄中已經有足夠資訊回答問題，就不需要再呼叫工具
'''        
        # 初始化訊息列表，第一條訊息是系統指令
        self.message = [SystemMessage(system_prompt)]

        # 將 LLM 的回應解析為純文字格式的工具
        self.str_parser = StrOutputParser()
       
    def chat_generator(self, text):
        """
        主對話生成函式（生成器形式）。
        逐步執行 LLM 回應與工具調用，並即時回傳每一步的結果。
        """
        # 將使用者的輸入加入訊息列表
        self.message.append(HumanMessage(text))        
        
        while True:
            # 呼叫 LLM，傳入完整訊息歷史
            
            final_ai_message = AIMessageChunk(content="")
            for chunk in self.llm_with_tools.stream(self.message):
                final_ai_message += chunk
                if hasattr(chunk, 'content') and chunk.content:
                    yield self.str_parser.invoke(chunk)
            
            response = final_ai_message
            
            # 將 LLM 回應加入訊息列表
            self.message.append(response)

            # 檢查 LLM 是否要求呼叫工具
            is_tools_call = False
            for tool_call in response.tool_calls:
                is_tools_call = True

                # 顯示 LLM 要執行的工具名稱與參數
                msg = f'[執行]: {tool_call["name"]}({tool_call["args"]})\n-----------\n' #完整訊息
                # msg = f'[執行]: {tool_call["name"]}()\n\n' #簡易訊息
                yield msg  # 使用 yield 讓結果能即時顯示在輸出中

                # 實際執行工具（根據工具名稱動態呼叫對應物件）
                tool_result = globals()[tool_call['name']].invoke(tool_call['args']) 

                # 顯示工具執行結果
                msg = f'[結果]: {tool_result}\n-----------\n'
                yield msg

                # 將工具執行結果封裝成 ToolMessage 回傳給 LLM
                tool_message = ToolMessage(
                    content=str(tool_result),          # 工具執行的文字結果
                    name=tool_call["name"],            # 工具名稱
                    tool_call_id=tool_call["id"],      # 工具呼叫 ID（讓 LLM 知道對應哪個呼叫）
                )
                # 將工具回傳結果加入訊息列表，提供 LLM 下一輪參考
                self.message.append(tool_message)
            
            # 若這一輪沒有任何工具呼叫，表示 LLM 已經生成最終回覆
            if is_tools_call == False:
                # 將 LLM 回應解析成純文字並輸出
                # yield self.str_parser.invoke(response)
                return  # 結束對話流程

    def chat(self, text, print_output=False):
        """
        封裝版對話函式。
        會收集 chat_generator 的所有輸出，並組合成完整的回覆字串。
        """
        msg = ''
        # 逐步取得 chat_generator 的產出內容
        for chunk in self.chat_generator(text):
            msg += f"{chunk}"
            if print_output:
                print(chunk, end='')
        # 回傳最終組合的對話內容
        return msg


In [24]:
bot = stream_chat_bot(llm, [get_qa])

In [25]:
for x in bot.chat_generator("你好！"):
    print(x, end='')

您好！很高興為您服務。請問有什麼可以協助您的嗎？

In [14]:
for x in bot.chat_generator("需要會員才能購物嗎？"):
    print(x, end='')

[執行]: get_qa({'question': '需要會員才能購物嗎？'})
-----------
[結果]: [Document(id='3208149b-fc68-4ac0-b2e1-32e5c4bfec06', metadata={'source': 'qa_data\\QA_會員註冊.txt'}, page_content='ClementShop 客戶服務詳細問答 - 會員註冊\n\nQ1: 如何加入 ClementShop 成為會員？\nA1: 請至 ClementShop 首頁右上角點選「註冊」，填寫基本資料後即可成為會員。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。\n\nQ2: 註冊會員是否需要收費？\nA2: ClementShop 會員註冊完全免費，無需支付任何費用。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。'), Document(id='83ea75d6-7fc7-43d4-954c-3f0cea9caa99', metadata={'source': 'qa_data\\QA_會員註冊.txt'}, page_content='Q3: 會員註冊需要提供哪些資料？\nA3: 您需提供姓名、電子郵件信箱及設定登入密碼，並確認服務條款。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。'), Document(id='3caa0

In [15]:
for x in bot.chat_generator("東西不滿意可以退貨嗎？"):
    print(x, end='')

[執行]: get_qa({'question': '東西不滿意可以退貨嗎？'})
-----------
[結果]: [Document(id='ac7482f8-e7f2-4ee3-a5ed-b27b8e37ebda', metadata={'source': 'qa_data\\QA_退貨.txt'}, page_content='Q10: 哪些商品不接受退貨？\nA10: 客製化、易腐壞或已拆封之商品恕不接受退貨。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。'), Document(id='818269b7-4a2b-4690-9d09-13aed2a83bd9', metadata={'source': 'qa_data\\QA_退貨.txt'}, page_content='Q4: 退貨商品需要保持什麼狀態？\nA4: 商品須保持全新未使用、包裝完整並附原始發票。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。\n\nQ5: 退貨申請審核多久？\nA5: 客服將於收到申請後 3 個工作天內完成審核。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。'), Document(id='f207e8f7-9007-49ec-b327-28d98f340900', metadata={'source': 'qa_data\\QA_退貨.txt'}, page_content='Q3: 退貨需要支付運費嗎？\nA3: 若為商品瑕疵或出貨錯誤，退貨運費由 ClementShop 負擔。 請注意，部分作業可能因例假日或物流延誤而稍有影響，敬請見諒。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 如遇特殊情況，

In [16]:
for x in bot.chat_generator("退貨有期限嗎？"):
    print(x, end='')

[執行]: get_qa({'question': '退貨有期限嗎？'})
-----------
[結果]: [Document(id='818269b7-4a2b-4690-9d09-13aed2a83bd9', metadata={'source': 'qa_data\\QA_退貨.txt'}, page_content='Q4: 退貨商品需要保持什麼狀態？\nA4: 商品須保持全新未使用、包裝完整並附原始發票。 我們建議您於完成操作後截圖保存，以便日後查詢紀錄或問題申訴使用。 所有流程皆可於會員中心即時查詢進度，讓您清楚掌握每一步狀態。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。\n\nQ5: 退貨申請審核多久？\nA5: 客服將於收到申請後 3 個工作天內完成審核。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。 若您在規定時間內未收到通知或確認信，請檢查垃圾郵件夾或聯繫客服協助處理。'), Document(id='071fbe2f-422b-4157-acc1-278d393069a4', metadata={'source': 'qa_data\\QA_退貨.txt'}, page_content='ClementShop 客戶服務詳細問答 - 退貨\n\nQ1: 如何申請退貨？\nA1: 請登入會員中心，於訂單詳情中點選「申請退貨」。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。 若申請內容涉及退款或帳號異動，處理時間將依銀行及系統作業日為準。 所有個人資料均受到嚴格保護，並依據個資法相關規範進行處理。 ClementShop 的系統會即時更新相關資訊，確保您的資料與訂單狀態保持最新。\n\nQ2: 退貨期限是多久？\nA2: 收到商品後七日內可申請退貨。 客服團隊提供每日 09:00 至 18:00 線上支援，將協助您解決各類問題。 若您在操作過程中遇到任何問題，建議您可先確認網路連線狀況，或重新整理頁面再試一次。 如遇特殊情況，我們將主動以電子郵件或簡訊通知您最新進度。'), Document(id='ac7482f8-e7f2-4ee3-a5ed-b27b8e37ebda', metadata={

In [17]:
for x in bot.chat_generator("退款會退到哪裡"):
    print(x, end='')

退款會退回到您原付款的帳戶或信用卡中。

In [19]:
for x in bot.chat_generator("機器學習的定義"):
    print(x, end='')

很抱歉，我無法回答與機器學習定義相關的問題。我的服務範圍僅限於 ClementShop 的平台、產品、訂單及客服流程等相關主題。如果您有任何關於 ClementShop 的問題，我很樂意為您服務。

In [20]:
for x in bot.chat_generator("你們家的商品比COCO還要貴，可以便宜一些嗎？"):
    print(x, end='')

很抱歉，我們的商品價格是根據多方面因素制定的，目前無法提供個別調整。感謝您的理解。